# MIND-small E0: EDA + R1–R5 + 3 graph snapshots



In [ ]:
from google.colab import files
uploaded = files.upload()  # Chọn mind_small_e0_graphs.zip (bộ code/project)
zip_name = next(iter(uploaded))
print('Uploaded project:', zip_name)


In [ ]:
import zipfile, os, glob
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall('/content')
project_dirs = glob.glob('/content/mind_small_e0_graphs')
assert project_dirs, 'Không tìm thấy thư mục mind_small_e0_graphs sau khi giải nén'
%cd /content/mind_small_e0_graphs
!pip -q install -r requirements.txt

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile

uploaded_data = files.upload()  # Chọn MINDsmall_train.zip và MINDsmall_dev.zip
data_root = Path("data")
data_root.mkdir(parents=True, exist_ok=True)

for archive_name in uploaded_data:
    lower = archive_name.lower()
    if not lower.endswith(".zip"):
        print(f"Bỏ qua file không phải ZIP: {archive_name}")
        continue
    if "train" in lower:
        folder_name = "MINDsmall_train"
    elif "dev" in lower:
        folder_name = "MINDsmall_dev"
    else:
        print(f"Không xác định được train/dev từ tên file: {archive_name}")
        continue
    destination = data_root / folder_name
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_name) as zf:
        root = destination.resolve()
        for member in zf.infolist():
            target = (destination / member.filename).resolve()
            if root not in target.parents and target != root:
                raise RuntimeError(f"ZIP có đường dẫn không an toàn: {member.filename}")
        zf.extractall(destination)
    print(f"Đã giải nén {archive_name} vào {destination}")

# Kiểm tra file quan trọng trước khi chạy pipeline
for folder_name in ["MINDsmall_train", "MINDsmall_dev"]:
    base = data_root / folder_name
    behaviors = list(base.rglob("behaviors.tsv"))
    news = list(base.rglob("news.tsv"))
    print(folder_name, "behaviors.tsv:", bool(behaviors), "news.tsv:", bool(news))
    if not behaviors or not news:
        raise FileNotFoundError(f"Thiếu behaviors.tsv hoặc news.tsv trong {base}. Kiểm tra ZIP bạn upload.")


## Bước 1 — chạy EDA và dựng các cạnh
Dataset đã được upload thủ công ở bước trên, nên **không thêm `--download`**. Lệnh này chạy EDA, A.2/A.3, R1–R5 và ba graph; tạm bỏ Louvain/NMI để có kết quả nền trước.


In [ ]:
!python run_mind_pipeline.py --data-dir data --output-dir outputs --skip-community


## Bước 2 — xem Bảng A.2 và A.3
Kiểm tra số đếm thật. Không chép số ước lượng vào báo cáo; đối chiếu Train cuối/Test với mốc trong proposal.

In [ ]:
import pandas as pd
display(pd.read_csv('outputs/A2_split_statistics.csv'))
display(pd.read_csv('outputs/A3_daily_eda.csv'))
display(pd.read_csv('outputs/history_consistency_summary.csv'))

## Bước 3 — tính SNA cộng đồng (Louvain modularity + NMI)
Nếu RAM/runtime đủ, chạy lại trên kết quả đã có nhưng không đặt `--skip-community`. Script đọc dataset đã upload trong `data/`, không tải dataset lại. Quá trình có thể chậm với graph lớn.


In [ ]:
!python run_mind_pipeline.py --data-dir data --output-dir outputs

## Bước 4 — tải outputs về máy
Thư mục `outputs/` chứa các bảng CSV, quan hệ R1–R5, 3 file graph edge CSV và kết quả SNA. Nên tải zip để không làm mất kết quả khi Colab ngắt phiên.

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/mind_small_outputs', 'zip', 'outputs')
files.download('/content/mind_small_outputs.zip')